# Imago UK explorer prep

This notebook covers prep of data required for the UK-wide embedding explorer of Imago. The source data is the officially released version for 2024 at:

> https://data.imago.ac.uk/datasets/google-satellite-embedding-v1-small-areas-2017-2024

In [42]:
import geopandas
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

- Data

In [2]:
db = geopandas.read_file('extracted-lsoa-geopackage_2024_output.gpkg')

ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/envs/gds/share/proj failed


## Pmtiles

- Inspect the layer to get names and info for Claude

In [4]:
! ogrinfo -so extracted-lsoa-geopackage_2024_output.gpkg

INFO: Open of `extracted-lsoa-geopackage_2024_output.gpkg'
      using driver `GPKG' successful.
1: 2024_output (Multi Polygon)


In [5]:
! ogrinfo -so extracted-lsoa-geopackage_2024_output.gpkg 2024_output

INFO: Open of `extracted-lsoa-geopackage_2024_output.gpkg'
      using driver `GPKG' successful.

Layer name: 2024_output
Geometry: Multi Polygon
Feature Count: 46844
Extent: (-68.839807, 5352.600000) - (655653.850000, 1220301.502000)
Layer SRS WKT:
PROJCRS["OSGB36 / British National Grid",
    BASEGEOGCRS["OSGB36",
        DATUM["Ordnance_Survey_of_Great_Britain_1936",
            ELLIPSOID["Airy 1830",6377563.396,299.3249646,
                LENGTHUNIT["metre",1]]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",4277]],
    CONVERSION["unnamed",
        METHOD["Transverse Mercator",
            ID["EPSG",9807]],
        PARAMETER["Latitude of natural origin",49,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8801]],
        PARAMETER["Longitude of natural origin",-2,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8802]],
        PARAMETER["Scale factor at natural origin",0.9996

- Write geoms and IDs to GeoJSON for `tippecanoe`

In [11]:
(
    db
    .to_crs('EPSG:4326')
    [['data_zone_code', 'geometry']]
    .to_file('uk_small_areas_2024.geojson')
)

- Build `.pmtiles`

In [15]:
%%time
! tippecanoe \
  -o uk_small_areas_2024.pmtiles \
  -l small_areas \
  --maximum-zoom=12 \
  --simplification=10 \
  --detect-shared-borders \
  --coalesce-densest-as-needed \
  --extend-zooms-if-still-dropping \
  --force \
  uk_small_areas_2024.geojson
! du -h uk_small_areas_2024.pmtiles

46844 features, 41479388 bytes of geometry and attributes, 515300 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
tile 4/7/5 size is 503924 with detail 12, >500000    
Going to try keeping the sparsest 79.38% of the features to make it fit
tile 4/7/5 size is 516287 with detail 12, >500000    
Going to try keeping the sparsest 61.50% of the features to make it fit
tile 5/15/10 size is 861295 with detail 12, >500000    
Going to try keeping the sparsest 46.44% of the features to make it fit
tile 5/15/10 size is 577630 with detail 12, >500000    
Going to try keeping the sparsest 32.16% of the features to make it fit
tile 6/31/20 size is 503580 with detail 12, >500000    
Going to try keeping the sparsest 79.43% of the features to make it fit
  99.9%  12/2040/1340  
30M	uk_small_areas_2024.pmtiles
CPU times: user 999 ms, sys: 302 ms, total: 1.3 s
Wall time: 1min


## Centroid lookup table

In [48]:
%%time
db_mercator = db.to_crs('EPSG:4326')
centroids = db_mercator.geometry.centroid
centroid_lon = centroids.x.values
centroid_lat = centroids.y.values
bounds = db_mercator.geometry.bounds  # minx, miny, maxx, maxy
bbox_minx = bounds["minx"].values
bbox_miny = bounds["miny"].values
bbox_maxx = bounds["maxx"].values
bbox_maxy = bounds["maxy"].values

centroids_table = pa.table({
    "area_code": db['data_zone_code'].values,
    "centroid_lon": centroid_lon,
    "centroid_lat": centroid_lat,
    "bbox_minx": bbox_minx,
    "bbox_miny": bbox_miny,
    "bbox_maxx": bbox_maxx,
    "bbox_maxy": bbox_maxy,
})
pq.write_table(
    centroids_table,
    "area_centroids.parquet",
    compression="snappy",
    row_group_size=1024,
)
! du -h area_centroids.parquet

<timed exec>:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



2.9M	area_centroids.parquet
CPU times: user 3.55 s, sys: 833 ms, total: 4.38 s
Wall time: 4.66 s


## Embeddings lookup tables

- Utils

In [40]:
K = 100
area_codes = db["data_zone_code"].values
band_cols = [f"band{i}_mean" for i in range(1, 65)]
embeddings = db[band_cols].values.astype(np.float64)

- Most similar

In [41]:
%%time
nn_sim = NearestNeighbors(
    n_neighbors=K + 1, metric="cosine", algorithm="brute"
)
nn_sim.fit(embeddings)
dist_sim, idx_sim = nn_sim.kneighbors(embeddings)
# Drop self-match (column 0)
idx_sim = idx_sim[:, 1:]

CPU times: user 1min 15s, sys: 4min 31s, total: 5min 46s
Wall time: 2min 8s


- Most dissimilar

In [45]:
%%time
n = len(area_codes)
idx_dis = np.empty((n, K), dtype=np.int64)

BATCH = 2000
for start in tqdm(range(0, n, BATCH), total=(n + BATCH - 1) // BATCH):
    end = min(start + BATCH, n)
    sims = cosine_similarity(embeddings[start:end], embeddings)

    for i_local, i_global in enumerate(range(start, end)):
        sims[i_local, i_global] = 2.0  # exclude self
        bot = np.argpartition(sims[i_local], K)[:K]
        idx_dis[i_global] = bot[np.argsort(sims[i_local, bot])]


100%|██████████| 24/24 [00:22<00:00,  1.07it/s]

CPU times: user 1min 15s, sys: 3min 4s, total: 4min 20s
Wall time: 22.5 s


- Write table out

In [47]:
%%time
columns = {
    "area_code": area_codes,
    "centroid_lon": centroid_lon,
    "centroid_lat": centroid_lat,
    "bbox_minx": bbox_minx,
    "bbox_miny": bbox_miny,
    "bbox_maxx": bbox_maxx,
    "bbox_maxy": bbox_maxy,
}

for j in range(K):
    columns[f"sim_{j+1:02d}"] = area_codes[idx_sim[:, j]]
    columns[f"dis_{j+1:02d}"] = area_codes[idx_dis[:, j]]

# Sort by area_code for efficient DuckDB range-request lookup
sort_idx = np.argsort(area_codes)
for key in columns:
    columns[key] = np.array(columns[key])[sort_idx]

neighbors_table = pa.table(columns)
pq.write_table(
    neighbors_table,
    "neighbors_2024.parquet",
    compression="snappy",
    row_group_size=1024,
)
! du -h neighbors_2024.parquet

44M	neighbors_2024.parquet
CPU times: user 1.48 s, sys: 142 ms, total: 1.62 s
Wall time: 1.95 s
